# VantageGro Retail Stock Insight
## Stage 3: Exploratory Data Analysis (EDA)

### Business & Strategic Objectives
Investigate historical demand patterns, store/regional performance, stockout vs overstock risks, promotion sensitivity, and SKU volatility.

### Core Business Questions Addressed
1. **Which product categories sell the most?**
2. **Which stores have the highest sales and customer demand?**
3. **When do stockouts and excess inventory occur?**
4. **Are promotions associated with statistically meaningful sales uplift?**
5. **How do prices and discounts relate to demand?**
6. **What seasonal and weather dynamics are present?**
7. **Which SKUs show high demand volatility requiring adaptive safety stocks?**

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (10, 5)
print('EDA Environment initialized successfully!')

### 1. Load Preprocessed Cleaned Dataset

In [ ]:
data_path = 'data/processed/vantagegro_cleaned.csv'
if not os.path.exists(data_path):
    data_path = '../data/processed/vantagegro_cleaned.csv'

df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
print(f'Cleaned Dataset: {len(df):,} records, {len(df.columns)} columns')
df.head()

### 2. Category & Store Sales vs Demand Performance

In [ ]:
# Category Performance
cat_df = df.groupby('Category').agg(
    total_sales=('Units Sold', 'sum'),
    total_demand=('Demand', 'sum'),
    avg_daily_demand=('Demand', 'mean'),
    avg_inventory=('Inventory Level', 'mean')
).reset_index().sort_values('total_demand', ascending=False)
print('=== CATEGORY PERFORMANCE ===')
display(cat_df)

plt.figure(figsize=(9, 4))
x = np.arange(len(cat_df))
plt.bar(x - 0.2, cat_df['total_demand'], 0.4, label='Customer Demand', color='#10b981')
plt.bar(x + 0.2, cat_df['total_sales'], 0.4, label='Units Sold', color='#3b82f6')
plt.title('Category Demand vs Units Sold', fontsize=12, fontweight='bold')
plt.xticks(x, cat_df['Category'])
plt.ylabel('Units')
plt.legend()
plt.tight_layout()
plt.show()

### 3. Store & Regional Dynamics

In [ ]:
store_df = df.groupby(['Store ID', 'Region']).agg(
    total_demand=('Demand', 'sum'),
    total_sales=('Units Sold', 'sum'),
    avg_inventory=('Inventory Level', 'mean')
).reset_index().sort_values('total_demand', ascending=False)
display(store_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=store_df, x='Store ID', y='total_demand', hue='Region', palette='viridis')
plt.title('Demand by Store ID and Region', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### 4. Stockout Risk vs Overstock Risk Quantification
- **Stockout Risk**: Inventory Level < Demand (demand cannot be met by stock)
- **Overstock Risk**: Inventory Level > 3 × Demand (holding excessive idle working capital)

In [ ]:
stockout_risk = (df['Inventory Level'] < df['Demand']).sum()
severe_stockout = (df['Inventory Level'] == 0).sum()
overstock_risk = (df['Inventory Level'] > (3 * df['Demand'])).sum()
balanced = len(df) - stockout_risk - overstock_risk

print(f'Stockout Risk Days:      {stockout_risk:,} ({stockout_risk/len(df)*100:.1f}%)')
print(f'Severe (Zero Stock):     {severe_stockout:,} ({severe_stockout/len(df)*100:.1f}%)')
print(f'Overstock Risk Days:     {overstock_risk:,} ({overstock_risk/len(df)*100:.1f}%)')
print(f'Balanced Stock Days:     {balanced:,} ({balanced/len(df)*100:.1f}%)')

plt.figure(figsize=(8, 4))
plt.bar(['Stockout Risk', 'Overstock Risk', 'Balanced'], [stockout_risk, overstock_risk, balanced], color=['#ef4444', '#f59e0b', '#10b981'])
plt.title('Inventory Health Risk Distribution', fontsize=11, fontweight='bold')
plt.ylabel('Records')
plt.tight_layout()
plt.show()

### 5. Promotional Lift & Discount Sensitivity

In [ ]:
promo_grp = df.groupby('Promotion')['Demand'].agg(['count', 'mean', 'median', 'std'])
print('=== PROMOTION DEMAND UPLIFT ===')
display(promo_grp)
uplift = (promo_grp.loc[1, 'mean'] - promo_grp.loc[0, 'mean']) / promo_grp.loc[0, 'mean'] * 100
print(f'Active Promotions generate an average Demand uplift of +{uplift:.1f}%!')

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x='Promotion', y='Demand', palette=['#94a3b8', '#10b981'])
plt.title('Demand Distribution: Standard (0) vs Promotional (1)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

### 6. Seasonality & Weather Variations

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
df.groupby('Seasonality')['Demand'].mean().plot(kind='bar', ax=ax1, color='#059669')
ax1.set_title('Mean Demand by Season', fontsize=10, fontweight='bold')
ax1.set_ylabel('Mean Demand')

df.groupby('Weather Condition')['Demand'].mean().plot(kind='bar', ax=ax2, color='#3b82f6')
ax2.set_title('Mean Demand by Weather', fontsize=10, fontweight='bold')
plt.suptitle('Environmental & Calendar Demand Drivers', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 7. Product Demand Volatility (Coefficient of Variation)
Segmenting SKUs to establish differentiated replenishment policies:
- High Volatility (CV > 0.45): Requires dynamic safety stocks
- Stable Demand (CV < 0.35): Suitable for standard periodic reordering

In [ ]:
prod_df = df.groupby(['Product ID', 'Category']).agg(
    mean_demand=('Demand', 'mean'),
    std_demand=('Demand', 'std'),
    total_demand=('Demand', 'sum')
).reset_index()
prod_df['cv_demand'] = (prod_df['std_demand'] / prod_df['mean_demand']).round(3)
prod_df['volatility'] = np.where(prod_df['cv_demand'] > 0.45, 'High Volatility', np.where(prod_df['cv_demand'] > 0.35, 'Moderate Volatility', 'Stable Demand'))
display(prod_df.sort_values('total_demand', ascending=False).head(10))

plt.figure(figsize=(9, 5))
sns.scatterplot(data=prod_df, x='mean_demand', y='cv_demand', hue='volatility', s=100, palette={'High Volatility': '#ef4444', 'Moderate Volatility': '#f59e0b', 'Stable Demand': '#10b981'})
for _, r in prod_df.iterrows():
    plt.text(r['mean_demand']+1, r['cv_demand'], r['Product ID'], fontsize=7)
plt.title('Product Volume vs Coefficient of Variation', fontsize=11, fontweight='bold')
plt.xlabel('Mean Daily Demand')
plt.ylabel('Coefficient of Variation')
plt.tight_layout()
plt.show()